# Competitive Intelligence Monitor

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Create semantic search** using `CREATE CORTEX SEARCH SERVICE`
2. **Index competitor documents** automatically
3. **Query with natural language** using `SEARCH()`
4. **Rank results by relevance** with similarity scores
5. **Track competitive activity** across multiple sources

---

## What You'll Build

A **competitive intelligence search system** with semantic document indexing:
- Semantic search across press releases, clinical data, publications
- Natural language queries ("What's Lilly's latest anticoagulant data?")
- Relevance-ranked results
- Threat assessment and activity tracking
- Strategic insights dashboard

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Approximately 15 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features Used

- `CREATE CORTEX SEARCH SERVICE` - Index documents for search
- `SEARCH()` - Semantic search queries
- Relevance scoring - Similarity metrics (0-1 scale)
- Attribute filtering - Filter by competitor, date, type
- `GENERATOR()` - Generar datos sintéticos competitor documents

Let's begin!

---

## Paso 1: Configuración del Entorno

### Cortex Search

**What is it?**  
A fully-managed search service that:
- Indexes text documents automatically
- Understands semantic meaning (not just keywords)
- Answers natural language questions
- Ranks results by relevance

### Why for Competitive Intelligence?

Track competitors by searching:
- Clinical trial publications
- Press releases and news articles
- FDA submissions and labels
- Conference presentations
- Social media and analyst reports

In [ ]:
CREATE DATABASE IF NOT EXISTS COMPETITIVE_INTEL_DB;
CREATE SCHEMA IF NOT EXISTS COMPETITIVE_INTEL_DB.PUBLIC;
USE SCHEMA COMPETITIVE_INTEL_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as db, CURRENT_SCHEMA() as schema, CURRENT_WAREHOUSE() as wh;

---

## Paso 2: Define Data Structure

### Tables

1. **`competitive_documents`** - Competitor publications, news, trials
2. **`search_queries`** - Track analyst questions and results

### Document Types

- **Clinical Trials**: Efficacy data, safety profiles
- **Press Releases**: Product launches, FDA approvals
- **Publications**: Peer-reviewed studies
- **News**: Analyst reports, market updates

In [ ]:
-- Competitive documents corpus
CREATE OR REPLACE TABLE competitive_documents (
    doc_id VARCHAR(50) PRIMARY KEY,
    doc_title VARCHAR(500),
    doc_text VARCHAR(10000),
    doc_type VARCHAR(50),  -- Clinical_Trial, Press_Release, Publication, News
    competitor VARCHAR(100),
    product_mentioned VARCHAR(100),
    publication_date DATE,
    source_url VARCHAR(500)
);

-- Search query log
CREATE OR REPLACE TABLE search_queries (
    query_id VARCHAR(50) PRIMARY KEY,
    query_text VARCHAR(1000),
    query_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    user_type VARCHAR(50)  -- Strategic_Planning, Medical_Affairs, Commercial
);

SELECT 'Tables created!' as status;

---

## Paso 3: Generar Datos Sintéticos Competitive Intelligence

Creating realistic competitor documents about diabetes drugs

In [ ]:
-- Generar competitive intelligence corpus
INSERT INTO competitive_documents
WITH competitor_content AS (
    SELECT * FROM (VALUES
        ('Lilly', 'Mounjaro', 'Clinical_Trial', 
         'SURPASS-2: Mounjaro Demonstrates Superior A1C Reduction vs. Semaglutide',
         'Phase 3 trial shows tirzepatide (Mounjaro) achieved mean A1C reduction of 2.3% vs. 1.9% for semaglutide 1mg at 40 weeks. Superior weight loss observed (11.2 kg vs. 6.7 kg). Gastrointestinal adverse events were similar between groups.'),
        ('Lilly', 'Mounjaro', 'Press_Release',
         'Lilly Announces FDA Approval of Mounjaro for Type 2 Diabetes',
         'Eli Lilly today announced the U.S. FDA has approved their anticoagulant for improving cardiovascular outcomes in adults with atrial fibrillation. This represents a new therapeutic option in AF management.'),
        ('Lilly', 'Trulicity', 'Publication',
         'Long-term Cardiovascular Outcomes with Dulaglutide: REWIND Trial Results',
         'The REWIND trial demonstrates dulaglutide reduces major adverse cardiovascular events by 12% vs. placebo in T2D patients. Benefits seen across broad patient population including those without established cardiovascular disease.'),
        ('AstraZeneca', 'Farxiga', 'Clinical_Trial',
         'DAPA-CKD: Dapagliflozin Reduces Kidney Disease Progression',
         'Farxiga demonstrated 39% relative risk reduction in composite of sustained eGFR decline, end-stage kidney disease, or renal/CV death in chronic kidney disease patients. Benefits observed regardless of diabetes status.'),
        ('AstraZeneca', 'Farxiga', 'News',
         'AstraZeneca Reports Strong Q3 Sales Growth for Farxiga',
         'Farxiga sales reached $1.6B in Q3, up 45% year-over-year, driven by heart failure and CKD indications. Company projects continued growth as awareness of cardiorenal benefits increases among clinicians.'),
        ('Boehringer Ingelheim', 'Jardiance', 'Publication',
         'Empagliflozin Reduces Heart Failure Hospitalization: EMPEROR-Reduced',
         'EMPEROR-Reduced trial shows empagliflozin reduces risk of cardiovascular death or heart failure hospitalization by 25% in patients with HFrEF. Benefit consistent regardless of diabetes status or ejection fraction.'),
        ('Boehringer Ingelheim', 'Jardiance', 'Press_Release',
         'Jardiance Approved for Expanded Heart Failure Indication',
         'FDA approves Jardiance for treatment of heart failure with reduced ejection fraction, making it the first SGLT2 inhibitor approved for HFrEF in patients with and without diabetes.'),
        ('Merck', 'Januvia', 'News',
         'Januvia Faces Generic Competition as Patent Expires',
         'Merck Januvia franchise declined 15% in recent quarter as generic sitagliptin enters market. Company focusing on next-generation diabetes portfolio including oral semaglutide partnerships.'),
        ('Sanofi', 'Lantus', 'Clinical_Trial',
         'Toujeo Demonstrates Non-Inferiority to Tresiba in Basal Insulin Study',
         'Head-to-head trial of Toujeo vs. Tresiba shows similar A1C reductions and hypoglycemia rates. Both ultra-long-acting insulins provide 24+ hour coverage with flexible dosing.'),
        ('Sanofi', 'Soliqua', 'Publication',
         'Fixed-Ratio Combination of Insulin Glargine and Lixisenatide in T2D',
         'LixiLan trial demonstrates Soliqua 100/33 achieves better glycemic control than basal insulin alone with favorable weight profile. Combination therapy reduces pill burden for patients.')
    ) t(competitor, product, doc_type, title, text)
)
SELECT 
    'DOC' || LPAD(SEQ4(), 5, '0') as doc_id,
    cc.title as doc_title,
    cc.text as doc_text,
    cc.doc_type,
    cc.competitor,
    cc.product as product_mentioned,
    DATEADD('day', -FLOOR(UNIFORM(30, 730, RANDOM())), CURRENT_DATE()) as publication_date,
    'https://clinicaltrials.gov/study/' || LPAD(FLOOR(UNIFORM(10000000, 99999999, RANDOM())), 8, '0') as source_url
FROM competitor_content cc,
TABLE(GENERATOR(ROWCOUNT => 100))
WHERE UNIFORM(0, 1, RANDOM()) > (SEQ4() / 110.0);

-- Verify data
SELECT 
    competitor,
    product_mentioned,
    doc_type,
    COUNT(*) as document_count
FROM competitive_documents
GROUP BY competitor, product_mentioned, doc_type
ORDER BY competitor, product_mentioned;

---

## Paso 4: Create Cortex Search Service

### Qué Vamos a Crear

**AI-powered semantic search service** that indexes competitive documents and enables natural language queries using vector embeddings and similarity search.

### Understanding `CREATE CORTEX SEARCH SERVICE`

Snowflake's **document indexing and semantic search engine** that transforms text into searchable vector embeddings:

```sql
CREATE CORTEX SEARCH SERVICE service_name
ON text_column                     -- Column containing documents/text to search
ATTRIBUTES attr1, attr2, ...      -- Metadata fields for filtering
WAREHOUSE = warehouse_name         -- Compute for indexing
TARGET_LAG = '1 hour'             -- How often to refresh index
AS SELECT columns FROM table;      -- Source data query
```

**Result**: Creates a persistent search service that automatically maintains an up-to-date semantic index

### Why CORTEX SEARCH for Competitive Intelligence?

`CREATE CORTEX SEARCH SERVICE` is ideal for document search because:

- **Semantic Understanding**: Finds documents by meaning, not just keywords
- **Automatic Indexing**: Continuously updates as new documents arrive
- **Vector Embeddings**: Converts text to mathematical representations for similarity
- **Fast Retrieval**: Sub-second search across millions of documents
- **Relevance Scoring**: Returns cosine similarity scores (0-1 scale)
- **Attribute Filtering**: Narrow results by competitor, date, document type, etc.
- **No ML Expertise Required**: Snowflake handles embedding models internally

### How It Works (Behind the Scenes)

When you create a Cortex Search Service, Snowflake:

1. **Reads Source Data**: Executes the `AS SELECT` query to get documents
2. **Generates Embeddings**: Converts each document's text into a vector (array of numbers)
   - Uses Snowflake's internal embedding model (similar to OpenAI's text-embedding-ada-002)
   - Each document becomes a 1536-dimensional vector
3. **Builds Vector Index**: Creates an optimized index structure (HNSW algorithm) for fast similarity search
4. **Stores Attributes**: Saves metadata fields (competitor, date, etc.) for filtering
5. **Monitors Changes**: Watches source table for new/updated documents
6. **Auto-Refreshes**: Reindexes based on `TARGET_LAG` setting

**Indexing Time**: 
- **Small datasets** (1K docs): ~30 seconds
- **Medio datasets** (100K docs): ~5 minutes
- **Large datasets** (1M docs): ~30 minutes

### What Are Vector Embeddings?

**Vector embeddings** are numerical representations of text that capture semantic meaning:

**Example**:
```
"Xarelto prevents stroke in AFib"  →  [0.23, -0.45, 0.78, ..., 0.12]  (1536 numbers)
"Rivaroxaban reduces thrombosis"  →  [0.25, -0.42, 0.81, ..., 0.09]  (1536 numbers)
"The weather is sunny today"        →  [-0.88, 0.12, -0.34, ..., 0.67]  (1536 numbers)
```

**Key Property**: **Semantically similar text has similar vectors**

**How Similarity Is Measured**:
- **Cosine Similarity**: Measures angle between two vectors
  - **1.0**: Identical meaning
  - **0.8-0.9**: Very similar
  - **0.6-0.7**: Somewhat similar
  - **< 0.5**: Different topics

**Why This Matters**:
- Traditional keyword search: "anticoagulant" finds only exact match
- Semantic search: "anticoagulant" also finds "rivaroxaban", "apixaban", "blood thinner"

### Key Parameters Explained

#### **1. `ON text_column`**

Specifies which column contains the text to index:

```sql
CREATE CORTEX SEARCH SERVICE my_search
ON doc_text        -- This column's content gets indexed
ATTRIBUTES ...
AS SELECT doc_id, doc_text, competitor FROM documents;
```

**What It Does**:
- Identifies the main searchable content
- This column's text is converted to embeddings
- Must be TEXT or VARCHAR type

**Best Practices**:
- **Short docs** (< 500 words): Index entire text
- **Long docs** (> 500 words): Consider chunking or summarizing
- **Multiple fields**: Concatenate if needed (`doc_title || ' ' || doc_body`)

#### **2. `ATTRIBUTES attr1, attr2, ...`**

Metadata fields for filtering search results:

```sql
CREATE CORTEX SEARCH SERVICE my_search
ON doc_text
ATTRIBUTES competitor, product, publication_date, doc_type
...
```

**What It Does**:
- Stores metadata alongside embeddings
- Enables filtering in search queries
- Not searchable semantically (exact match only)

**Use Cases**:
- **Filter by source**: `competitor = 'Lilly'`
- **Filter by date**: `publication_date > '2024-01-01'`
- **Filter by type**: `doc_type = 'clinical_trial'`

**Performance Tip**: Attributes are stored as-is, not indexed separately. Use for low-cardinality fields.

#### **3. `WAREHOUSE = warehouse_name`**

Compute resource for indexing and refreshing:

```sql
CREATE CORTEX SEARCH SERVICE my_search
...
WAREHOUSE = COMPUTE_WH   -- Use existing warehouse
```

**What It Does**:
- Powers initial indexing job
- Handles incremental refresh jobs
- Billed per second when indexing runs

**Sizing Guidance**:
- **< 10K docs**: X-Small warehouse
- **10K-100K docs**: Small warehouse
- **> 100K docs**: Medio+ warehouse

**Note**: Warehouse only runs during indexing/refresh, not during searches

#### **4. `TARGET_LAG = 'duration'`**

How often to refresh the index with new data:

```sql
CREATE CORTEX SEARCH SERVICE my_search
...
TARGET_LAG = '1 hour'   -- Refresh hourly
```

**What It Does**:
- Sets maximum acceptable lag between source data and search index
- Snowflake automatically schedules refresh jobs
- Balances freshness vs. compute usage

**Common Settings**:
- **Real-time**: `'1 minute'` - For critical alerts
- **Near real-time**: `'15 minutes'` - For dashboards
- **Batch**: `'1 hour'` or `'1 day'` - For reporting

**Trade-off**: More frequent refreshes = more compute usage

#### **5. `AS SELECT ...`**

Source data query that feeds the search service:

```sql
CREATE CORTEX SEARCH SERVICE my_search
ON doc_text
ATTRIBUTES competitor, product
AS SELECT 
    doc_id,
    doc_text,          -- Must include text column
    competitor,        -- Must include all attributes
    product
FROM competitive_documents
WHERE is_relevant = TRUE;   -- Can filter source data
```

**What It Does**:
- Defines which rows to index
- Can filter, transform, or join data
- Must include text column + all attributes

**Best Practices**:
- **Filter early**: Don't index irrelevant data
- **Join if needed**: Can combine multiple tables
- **Transform text**: Clean/normalize before indexing

**Example - Multi-Table**:
```sql
CREATE CORTEX SEARCH SERVICE combined_search
ON full_text
ATTRIBUTES source_type, date
AS SELECT 
    press_release_id as doc_id,
    title || ' ' || body as full_text,
    'press_release' as source_type,
    published_date as date
FROM press_releases
UNION ALL
SELECT 
    trial_id,
    abstract || ' ' || results as full_text,
    'clinical_trial',
    completion_date
FROM clinical_trials;
```

### Search Service Lifecycle

**1. Creation**:
```sql
CREATE CORTEX SEARCH SERVICE my_search ...
-- Status: CREATING (wait 30 sec - 30 min)
```

**2. Ready**:
```sql
SHOW CORTEX SEARCH SERVICES LIKE 'MY_SEARCH';
-- Status: READY (can now query)
```

**3. Querying**:
```sql
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW('my_search', '{"query": "..."}');
-- Returns: Relevant documents + scores
```

**4. Automatic Refresh**:
```sql
-- Snowflake automatically runs:
-- - Check source table for changes
-- - Re-index new/modified documents
-- - Update search index
-- Frequency based on TARGET_LAG
```

**5. Monitoring**:
```sql
-- Check last refresh time
DESCRIBE CORTEX SEARCH SERVICE my_search;

-- View indexing history
SELECT * FROM TABLE(INFORMATION_SCHEMA.CORTEX_SEARCH_SERVICE_HISTORY(
    SERVICE_NAME => 'my_search'
));
```

**6. Deletion**:
```sql
DROP CORTEX SEARCH SERVICE my_search;
-- All embeddings and indexes are removed
```

### Real-World Example: Competitive Intelligence

**Scenario**: Monitor 50,000 competitor documents (press releases, trials, SEC filings)

**Implementation**:
```sql
CREATE OR REPLACE CORTEX SEARCH SERVICE competitive_intel_search
ON full_document_text                    -- Main searchable content
ATTRIBUTES 
    document_title,                      -- Metadata for display
    document_type,                       -- Filter: press_release, trial, filing
    competitor_name,                     -- Filter: Lilly, Sanofi, AstraZeneca
    product_mentioned,                   -- Filter: Mounjaro, Trulicity
    publication_date,                    -- Filter: recent vs. historical
    geographic_market                    -- Filter: US, EU, APAC
WAREHOUSE = COMPETITIVE_INTEL_WH         -- Medio warehouse (4 credits/hr)
TARGET_LAG = '15 minutes'               -- Near real-time for alerts
AS 
SELECT 
    doc_id,
    doc_title as document_title,
    doc_type as document_type,
    competitor as competitor_name,
    product as product_mentioned,
    pub_date as publication_date,
    market as geographic_market,
    -- Combine fields for richer search
    doc_title || '\n\n' || doc_summary || '\n\n' || doc_full_text as full_document_text
FROM competitive_documents
WHERE 
    is_deleted = FALSE                   -- Exclude deleted docs
    AND language = 'en'                  -- English only
    AND relevance_score > 0.5;          -- Pre-filter noise
```

**Result**:
- **Initial indexing**: ~10 minutes (50K docs on Medio warehouse)

- **Ongoing**: Refreshes every 15 min (only if new data)
- **Query latency**: < 500ms for semantic search

**Query Example**:
```sql
-- Find documents about Lilly's cardiovascular outcomes
SELECT r.value:doc_title::STRING as title,
       r.value:"@search_score"::FLOAT as relevance
FROM TABLE(
    FLATTEN(
        PARSE_JSON(
            SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
                'competitive_intel_search',
                '{"query": "Lilly cardiovascular outcomes anticoagulant",
                  "columns": ["document_title", "publication_date"],
                  "filter": {"@eq": {"competitor_name": "Lilly"}}}'
            )
        )['results']
    )
) r
WHERE r.value:"@search_score"::FLOAT > 0.7  -- Alto relevance only
ORDER BY r.value:"@search_score"::FLOAT DESC
LIMIT 10;
```

### Why This Matters for Pharma

**Old Way** (keyword search):
- Query: "anticoagulant efficacy"
- Finds: Only exact matches for "anticoagulant" AND "efficacy"
- Misses: "semaglutide improves glucose control" (no keywords match!)

**CORTEX SEARCH Way** (semantic):
- Query: "anticoagulant efficacy"
- Finds: All documents about anticoagulant effects, even if worded differently
- Includes: "tirzepatide reduces HbA1c", "dulaglutide effectiveness", "semaglutide outcomes"
- Relevance scored: Most relevant documents ranked first

**Impacto de Negocio**:
- **Coverage**: Find **90%+ more relevant** documents vs. keyword search
- **Speed**: Search 1M documents in < 1 second
- **Actionable**: Identify competitor moves within 15 minutes of publication

### Technical Details

**Embedding Model**:
- Snowflake uses proprietary embedding model
- Similar quality to OpenAI's text-embedding-ada-002
- Optimized for business/technical documents
- Automatically handles up to **8,000 tokens per document** (~6,000 words)

**Vector Index Algorithm**:
- **HNSW** (Hierarchical Navigable Small World)
- Approximate nearest neighbor search
- Trade-off: 99%+ recall with 10-100x speedup vs. exact search

**Storage**:
- Each document: ~6 KB (1536 dimensions × 4 bytes/float)
- 1M documents: ~6 GB storage


**Query Performance**:
- **< 100K docs**: < 100ms
- **< 1M docs**: < 500ms
- **< 10M docs**: < 2 seconds
- Scales horizontally (no performance degradation)

**Supported Languages**:
- English, Spanish, French, German, Italian, Portuguese, Dutch, Russian, Chinese, Japanese, Korean
- Multi-language support: Index documents in multiple languages simultaneously

**Limitations**:
- **Max document size**: 8,000 tokens (~6,000 words)
- **Max attributes**: 20 fields
- **Max service size**: 10M documents (contact Snowflake for larger)
- **Refresh frequency**: Minimum `TARGET_LAG = '1 minute'`

### Best Practices

**DO**:
-  Include descriptive attributes (date, type, source)
-  Use `TARGET_LAG` appropriate for use case (don't over-refresh)
-  Pre-filter source data (don't index everything)
-  Monitor service status after creation
-  Test queries before deploying to production

**DON'T**:
-  Index extremely long documents (> 6,000 words) without chunking
-  Use attributes for high-cardinality fields (e.g., unique IDs)
-  Set `TARGET_LAG` too low for static data (wastes credits)
-  Index data you don't need (unnecessary storage)
-  Forget to grant USAGE on service to users

### Common Patterns

**Pattern 1: News Monitoring**:
```sql
CREATE CORTEX SEARCH SERVICE news_monitor
ON article_text
ATTRIBUTES published_date, source, topic
TARGET_LAG = '5 minutes'   -- Near real-time
AS SELECT * FROM news_feed WHERE language = 'en';
```

**Pattern 2: Document Library**:
```sql
CREATE CORTEX SEARCH SERVICE doc_library
ON document_content
ATTRIBUTES doc_type, department, created_by
TARGET_LAG = '1 day'       -- Batch refresh
AS SELECT * FROM documents WHERE is_published = TRUE;
```

**Pattern 3: Customer Support**:
```sql
CREATE CORTEX SEARCH SERVICE support_kb
ON solution_text
ATTRIBUTES category, product, last_updated
TARGET_LAG = '1 hour'
AS SELECT * FROM knowledge_base WHERE status = 'approved';
```

### Troubleshooting

**Issue**: Service stuck in CREATING status
**Solution**: Check warehouse is running, source table has data

**Issue**: Search returns no results
**Solution**: Verify service status is READY, check query syntax

**Issue**: Bajo relevance scores (all < 0.5)
**Solution**: Query too vague - be more specific, or source data doesn't match domain

**Issue**: Alto compute usage
**Solution**: Increase `TARGET_LAG`, reduce document count, or use smaller warehouse

In [ ]:
-- Create Cortex Search service on competitive documents
CREATE OR REPLACE CORTEX SEARCH SERVICE competitive_intel_search
ON doc_text
ATTRIBUTES doc_title, doc_type, competitor, product_mentioned, publication_date
WAREHOUSE = COMPUTE_WH
TARGET_LAG = '1 hour'
AS
SELECT 
    doc_id,
    doc_text,
    doc_title,
    doc_type,
    competitor,
    product_mentioned,
    publication_date
FROM competitive_documents;

-- Verify search service is created
SHOW CORTEX SEARCH SERVICES LIKE 'COMPETITIVE_INTEL_SEARCH';

-- Wait for service to be ready (status should show 'READY')
-- If status is 'PROVISIONING' or 'INDEXING', wait 1-2 minutes before running next cells

SELECT 'Search service created! Check status above - must be READY before proceeding.' as status;

---

## Paso 5: Perform Semantic Searches

### Using `SNOWFLAKE.CORTEX.SEARCH_PREVIEW()`

**Syntax**:
```sql
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'service_name',
        '{"query": "your question", 
          "columns": ["col1", "col2"], 
          "limit": 10}'
    )
)['results'] as results;
```

**Key Points**:
- The second parameter is a **JSON string** containing query options
- Wrap the call in `PARSE_JSON()` and extract `['results']`
- Use `LATERAL FLATTEN()` to unpack the results array

Ask questions in plain English and get semantically relevant results!

In [ ]:
-- Search for anticoagulant competitor data
WITH search_results AS (
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'COMPETITIVE_INTEL_DB.PUBLIC.COMPETITIVE_INTEL_SEARCH',
            '{"query": "What are the latest anticoagulant efficacy results compared to rivaroxaban?", 
              "columns": ["competitor", "product_mentioned", "doc_title", "doc_type"],
              "limit": 10}'
        )
    )['results'] as results
)
SELECT 
    r.value:competitor::STRING as competitor,
    r.value:product_mentioned::STRING as product_mentioned,
    r.value:doc_title::STRING as doc_title,
    r.value:doc_type::STRING as doc_type,
    r.value:publication_date::DATE as publication_date,
    ROUND(r.value:"@scores".cosine_similarity::FLOAT, 3) as relevance_score
FROM search_results,
LATERAL FLATTEN(input => search_results.results) r
ORDER BY relevance_score DESC
LIMIT 10;

-- Search for cardiovascular outcomes
WITH search_results AS (
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'COMPETITIVE_INTEL_DB.PUBLIC.COMPETITIVE_INTEL_SEARCH',
            '{"query": "Which competitors have cardiovascular outcomes data?",
              "columns": ["competitor", "product_mentioned", "doc_title", "doc_type"],
              "limit": 10}'
        )
    )['results'] as results
)
SELECT 
    r.value:competitor::STRING as competitor,
    r.value:product_mentioned::STRING as product_mentioned,
    r.value:doc_title::STRING as doc_title,
    r.value:publication_date::DATE as publication_date,
    ROUND(r.value:"@scores".cosine_similarity::FLOAT, 3) as relevance_score
FROM search_results,
LATERAL FLATTEN(input => search_results.results) r
ORDER BY relevance_score DESC
LIMIT 10;

-- Search for market access challenges
WITH search_results AS (
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'COMPETITIVE_INTEL_DB.PUBLIC.COMPETITIVE_INTEL_SEARCH',
            '{"query": "What are competitors saying about pricing and market access?",
              "columns": ["competitor", "product_mentioned", "doc_title", "doc_type"],
              "limit": 10}'
        )
    )['results'] as results
)
SELECT 
    r.value:competitor::STRING as competitor,
    r.value:product_mentioned::STRING as product_mentioned,
    r.value:doc_title::STRING as doc_title,
    ROUND(r.value:"@scores".cosine_similarity::FLOAT, 3) as relevance_score
FROM search_results,
LATERAL FLATTEN(input => search_results.results) r
ORDER BY relevance_score DESC
LIMIT 10;

---

## Paso 6: Build Search Query Analytics

Track what questions analysts are asking and save results for trend analysis

In [ ]:
-- Log search queries for analytics
INSERT INTO search_queries (query_id, query_text, user_type)
SELECT UUID_STRING(), 'What is Lilly Mounjaro clinical trial data?', 'Medical_Affairs'
UNION ALL
SELECT UUID_STRING(), 'Compare SGLT2 inhibitor cardiovascular benefits', 'Strategic_Planning'
UNION ALL
SELECT UUID_STRING(), 'Latest FDA approvals in diabetes space', 'Commercial'
UNION ALL
SELECT UUID_STRING(), 'Competitor pricing strategies', 'Market_Access'
UNION ALL
SELECT UUID_STRING(), 'Generic competition timeline', 'Strategic_Planning';

-- Example: Search for Mounjaro clinical data
-- Note: SEARCH_PREVIEW requires constant string literals, so we can't dynamically
-- query based on the search_queries table. For dynamic queries, use the Python API.
CREATE OR REPLACE TABLE competitive_search_results AS
WITH mounjaro_search AS (
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'COMPETITIVE_INTEL_DB.PUBLIC.COMPETITIVE_INTEL_SEARCH',
            '{"query": "What is Lilly Mounjaro clinical trial data?", 
              "columns": ["competitor", "product_mentioned", "doc_title", "doc_type"],
              "limit": 5}'
        )
    )['results'] as results
),
sglt2_search AS (
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'COMPETITIVE_INTEL_DB.PUBLIC.COMPETITIVE_INTEL_SEARCH',
            '{"query": "Compare SGLT2 inhibitor cardiovascular benefits", 
              "columns": ["competitor", "product_mentioned", "doc_title", "doc_type"],
              "limit": 5}'
        )
    )['results'] as results
)
SELECT 
    'Mounjaro clinical data' as query_category,
    r.value:competitor::STRING as competitor,
    r.value:product_mentioned::STRING as product_mentioned,
    r.value:doc_title::STRING as doc_title,
    r.value:doc_type::STRING as doc_type,
    ROUND(r.value:"@scores".cosine_similarity::FLOAT, 3) as relevance_score
FROM mounjaro_search,
LATERAL FLATTEN(input => mounjaro_search.results) r
UNION ALL
SELECT 
    'SGLT2 cardiovascular' as query_category,
    r.value:competitor::STRING as competitor,
    r.value:product_mentioned::STRING as product_mentioned,
    r.value:doc_title::STRING as doc_title,
    r.value:doc_type::STRING as doc_type,
    ROUND(r.value:"@scores".cosine_similarity::FLOAT, 3) as relevance_score
FROM sglt2_search,
LATERAL FLATTEN(input => sglt2_search.results) r
ORDER BY query_category, relevance_score ASC;

-- View results
SELECT 
    query_category,
    competitor,
    product_mentioned,
    doc_title,
    relevance_score
FROM competitive_search_results
LIMIT 20;

---

## Paso 7: Competitive Positioning Analysis

Aggregate insights by competitor and product

In [ ]:
-- Competitive landscape summary
CREATE OR REPLACE TABLE competitive_landscape AS
WITH doc_summary AS (
    SELECT 
        competitor,
        product_mentioned,
        COUNT(*) as total_documents,
        COUNT(CASE WHEN doc_type = 'Clinical_Trial' THEN 1 END) as clinical_trials,
        COUNT(CASE WHEN doc_type = 'Press_Release' THEN 1 END) as press_releases,
        COUNT(CASE WHEN doc_type = 'Publication' THEN 1 END) as publications,
        MAX(publication_date) as latest_activity,
        DATEDIFF('day', latest_activity, CURRENT_DATE()) as days_since_activity
    FROM competitive_documents
    GROUP BY competitor, product_mentioned
),
activity_score AS (
    SELECT 
        *,
        -- Activity score (more recent = higher)
        CASE 
            WHEN days_since_activity < 30 THEN 'High Activity'
            WHEN days_since_activity < 180 THEN 'Moderate Activity'
            ELSE 'Low Activity'
        END as activity_level,
        -- Threat level based on clinical trial pipeline
        CASE 
            WHEN clinical_trials >= 3 AND days_since_activity < 90 THEN '🔴 High Threat'
            WHEN clinical_trials >= 2 OR (press_releases >= 2 AND days_since_activity < 60) THEN '🟡 Moderate Threat'
            ELSE '🟢 Low Threat'
        END as threat_assessment
    FROM doc_summary
)
SELECT * FROM activity_score
ORDER BY 
    CASE threat_assessment
        WHEN '🔴 High Threat' THEN 1
        WHEN '🟡 Moderate Threat' THEN 2
        ELSE 3
    END,
    days_since_activity;

-- View competitive landscape
SELECT 
    competitor,
    product_mentioned,
    total_documents,
    clinical_trials,
    latest_activity,
    activity_level,
    threat_assessment
FROM competitive_landscape
ORDER BY threat_assessment, competitor;

---

## Paso 8: Interactive Competitive Intelligence Dashboard

Search documents and visualize competitive landscape

In [ ]:
import streamlit as st
import pandas as pd
import warnings
import json as pyjson
from snowflake.snowpark.context import get_active_session

# Suppress warnings
warnings.filterwarnings('ignore', message='.*vegalite type.*')

session = get_active_session()

st.title("🔍 Competitive Intelligence Monitor")
st.markdown("### Semantic search across competitor documents")

# Key metrics
total_docs = session.sql("SELECT COUNT(*) as cnt FROM competitive_documents").collect()[0]['CNT']
competitors = session.sql("SELECT COUNT(DISTINCT competitor) as cnt FROM competitive_documents").collect()[0]['CNT']
recent_activity = session.sql("""
    SELECT COUNT(*) as cnt 
    FROM competitive_documents 
    WHERE publication_date >= DATEADD('day', -30, CURRENT_DATE())
""").collect()[0]['CNT']

col1, col2, col3 = st.columns(3)
with col1:
    st.metric("Total Documents", int(total_docs))
with col2:
    st.metric("Competitors Tracked", int(competitors))
with col3:
    st.metric("Recent Activity (30d)", int(recent_activity))

# Search interface
st.markdown("---")
st.subheader("🔎 Semantic Search")

search_query = st.text_input(
    "Ask a question about competitors",
    placeholder="e.g., What are the latest anticoagulant clinical trial results?"
)

if search_query:
    try:
        # Build JSON query string
        query_obj = {
            "query": search_query,
              "columns": ["competitor", "product_mentioned", "doc_title", "doc_type"],
            "limit": 10
        }
        query_json = pyjson.dumps(query_obj).replace("'", "''")
        
        results_df = session.sql(f"""
            WITH search_results AS (
                SELECT PARSE_JSON(
                    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
                        'COMPETITIVE_INTEL_DB.PUBLIC.COMPETITIVE_INTEL_SEARCH',
                        '{query_json}'
                    )
                )['results'] as results
            )
            SELECT 
                r.value:competitor::STRING as competitor,
                r.value:product_mentioned::STRING as product_mentioned,
                r.value:doc_title::STRING as doc_title,
                r.value:doc_type::STRING as doc_type,
                r.value:publication_date::DATE as publication_date,
                ROUND(r.value:"@scores".cosine_similarity::FLOAT, 3) as relevance
            FROM search_results,
            LATERAL FLATTEN(input => search_results.results) r
            ORDER BY relevance DESC
            LIMIT 10
        """).to_pandas()
        
        if len(results_df) > 0:
            st.dataframe(results_df, use_container_width=True, hide_index=True)
        else:
            st.info("No results found. Try a different query.")
    except Exception as e:
        st.error(f"Search error: {str(e)}")
        st.info("Make sure the Cortex Search service is fully indexed.")

# Competitive landscape
st.markdown("---")
st.subheader("🗺️ Competitive Landscape")

landscape = session.sql("""
    SELECT competitor, product_mentioned, total_documents, 
           clinical_trials, activity_level, threat_assessment
    FROM competitive_landscape
    ORDER BY threat_assessment, competitor
""").to_pandas()

st.dataframe(landscape, use_container_width=True, hide_index=True)

# Activity by competitor
st.markdown("---")
st.subheader("📊 Document Activity by Competitor")

activity = session.sql("""
    SELECT competitor, doc_type, COUNT(*) as count
    FROM competitive_documents
    GROUP BY competitor, doc_type
""").to_pandas()

if len(activity) > 0:
    try:
        pivot = activity.pivot(index='COMPETITOR', columns='DOC_TYPE', values='COUNT').fillna(0)
        if len(pivot) > 0 and len(pivot.columns) > 0:
            # Transpose so competitors are columns
            chart_data = pivot.T
            st.bar_chart(chart_data)
        else:
            st.info("No document type distribution available.")
    except Exception as e:
        st.warning(f"Unable to create chart: {str(e)}")
        st.dataframe(activity, use_container_width=True, hide_index=True)
else:
    st.info("No activity data available.")
    chart_data = chart_data.set_index('Competitor')
    st.bar_chart(chart_data)

# Recent updates
st.markdown("---")
st.subheader("📰 Recent Competitor Activity")

recent = session.sql("""
    SELECT competitor, product_mentioned, doc_title, doc_type, publication_date
    FROM competitive_documents
    WHERE publication_date >= DATEADD('day', -90, CURRENT_DATE())
    ORDER BY publication_date DESC
    LIMIT 15
""").to_pandas()

st.dataframe(recent, use_container_width=True, hide_index=True)

# Download
st.markdown("---")
csv = landscape.to_csv(index=False)
st.download_button("📥 Download Competitive Intelligence Report", data=csv, file_name="competitive_intel.csv", mime="text/csv")

---

##  Tutorial Complete!

### What You've Learned

1.  Created **Cortex Search Service** for document indexing
2.  Performed **semantic searches** with natural language
3.  Tracked competitor activity across multiple sources
4.  Built competitive intelligence dashboards
5.  Analyzed threat levels and activity patterns
6.  Generated strategic insights from unstructured data

### Key Cortex Search Features

**`CREATE CORTEX SEARCH SERVICE`**
- Automatically indexes text documents
- Understands semantic meaning (not just keywords)
- Continuously updated as data changes
- Fully managed (no infrastructure to maintain)

**`SEARCH()`**
- Natural language queries
- Returns ranked results by relevance
- Filters by attributes (competitor, date, etc.)

### Competitive Intelligence Insights

- **Alto-value sources**: Clinical trials, FDA filings, peer-reviewed publications
- **Activity tracking**: Monitor launch timelines and pipeline progress
- **Threat assessment**: Prioritize competitive responses
- **Strategic planning**: Inform product positioning and messaging

### Business Applications

- **Medical Affairs**: Track clinical evidence and publications
- **Strategic Planning**: Monitor pipeline and M&A activity
- **Commercial Teams**: Understand competitive positioning
- **Market Access**: Identify payer coverage and pricing trends

### Next Steps

- Integrate real competitor data feeds
- Add automated daily monitoring and alerts
- Build comparison views (head-to-head efficacy tables)
- Create executive briefings from search results
- Expand to include conference presentations and social media

### Resources

- [Cortex Search](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)
- [CREATE CORTEX SEARCH SERVICE](https://docs.snowflake.com/en/sql-reference/sql/create-cortex-search-service)
- [Cortex Search Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/query-cortex-search-service)

---

**Congratulations!** You've built a complete competitive intelligence system with Cortex Search.

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `COMPETITIVE_INTEL_DB` database (and all tables/search services within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS COMPETITIVE_INTEL_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
